# 2장 1강: t-검정의 이론과 가정 — 실습문제

## 실습 목표

- 평균 차이와 표준오차를 이용해 t통계량을 직접 계산할 수 있다.
- 단일표본 t검정으로 하나의 표본평균과 기준값을 비교할 수 있다.
- 독립성·정규성·등분산성을 확인한 뒤 독립표본 t검정을 수행할 수 있다.
- 독립표본과 대응표본 상황을 구분하고 적절한 함수를 선택할 수 있다.
- p-value를 유의수준과 비교하여 검정 결과를 올바르게 해석할 수 있다.

## 실습 환경 / 데이터

- Python
- NumPy, pandas
- scipy.stats
- `ames_housing.csv`

| 컬럼 | 의미 |
|---|---|
| `SalePrice` | 주택 판매가격 |
| `KitchenQual` | 주방 품질 |

> 모든 검정은 양측검정이며 유의수준 `α = 0.05`를 사용합니다.  
> Ames 표본 추출에는 지정된 `random_state`를 사용해 결과를 재현합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 전체 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.
import pandas as pd

df = pd.read_csv(r'C:\Machine_Learning_worksheet\Data\ames_housing.csv', encoding = 'utf-8-sig')

---

## 필수 1. t통계량 직접 계산과 단일표본 t검정

### 문제 1-1. 평균 판매가격은 180,000달러와 다른가?

#### 문제 설명

Ames 주택 30개를 표본으로 추출하여 평균 판매가격이 기준값 180,000달러와 다른지 확인합니다. 먼저 t통계량을 직접 계산한 뒤 `ttest_1samp()` 결과와 비교하세요.

#### 요구사항

1. `SalePrice`에서 `n=30`, `random_state=2`로 표본을 추출하여 `sale_sample`에 저장하세요.
2. 표본 수, 표본평균, 표본표준편차를 출력하세요.
3. `stats.sem()`을 이용해 표준오차를 계산하세요.
4. 다음 식으로 t통계량을 직접 계산하세요.  
   `t = (표본평균 - 기준값) / 표준오차`
5. Shapiro-Wilk 검정으로 표본의 정규성을 확인하세요.
6. 다음 가설을 작성하세요.
   - H₀: 모집단 평균 판매가격은 180,000달러이다.
   - H₁: 모집단 평균 판매가격은 180,000달러가 아니다.
7. `stats.ttest_1samp()`로 단일표본 t검정을 수행하세요.
8. 직접 계산한 t통계량과 함수가 반환한 t통계량을 비교하세요.
9. p-value를 이용해 귀무가설 기각 여부를 판단하세요.

#### 해석 질문

**Q1.** t통계량은 평균 차이와 표준오차를 어떻게 이용한 값인가요?  
**Q2.** 같은 평균 차이라면 표준오차가 작아질수록 t통계량의 절댓값은 어떻게 변하나요?  
**Q3.** 직접 계산한 t통계량과 `ttest_1samp()`의 t통계량은 일치하나요?  
**Q4.** 검정 결과 평균 판매가격이 180,000달러와 다르다고 판단할 수 있나요?

#### 제출 결과

- 기술통계량과 정규성 결과
- 직접 계산한 표준오차와 t통계량
- 단일표본 t검정 결과
- 귀무가설 판단과 해석
- Q1~Q4 답변

In [3]:
# 필수 1 코드를 작성하세요.

import numpy as np
import pandas as pd
from scipy import stats


# 1. SalePrice에서 n=30, random_state=2로 표본 추출
sale_sample = df["SalePrice"].dropna().sample(n=30, random_state=2)

# 2. 표본 수, 표본평균, 표본표준편차
n = len(sale_sample)
sample_mean = sale_sample.mean()
sample_std = sale_sample.std()  # pandas 기본값 ddof=1 (표본표준편차)
print(f"표본 수: {n}")
print(f"표본평균: {sample_mean:.4f}")
print(f"표본표준편차: {sample_std:.4f}")

# 3. 표준오차
se = stats.sem(sale_sample)  # 기본값 ddof=1 → s / sqrt(n)
print(f"표준오차: {se:.4f}")

# 4. t통계량 직접 계산
mu0 = 180000
t_manual = (sample_mean - mu0) / se
print(f"직접 계산한 t통계량: {t_manual:.4f}")

# 5. Shapiro-Wilk 정규성 검정
# H0: 표본이 정규분포를 따르는 모집단에서 추출되었다
alpha = 0.05
shapiro_stat, shapiro_p = stats.shapiro(sale_sample)
print(f"\nShapiro-Wilk 통계량: {shapiro_stat:.4f}, p-value: {shapiro_p:.4f}")
if shapiro_p >= alpha:
    print("p-value >= 0.05 → 정규성 가정을 기각하지 못함 (정규성 충족으로 봄)")
else:
    print("p-value < 0.05 → 정규성 가정 위반")

# 6. 가설
# H0: 모집단 평균 판매가격은 180,000달러이다. (μ = 180,000)
# H1: 모집단 평균 판매가격은 180,000달러가 아니다. (μ ≠ 180,000)

# 7. 단일표본 t검정
t_func, p_value = stats.ttest_1samp(sale_sample, popmean=mu0, alternative="two-sided")
print(f"\nttest_1samp t통계량: {t_func:.4f}")
print(f"p-value: {p_value:.4f}")

# 8. 직접 계산한 t통계량과 함수 결과 비교
print(f"\n직접 계산: {t_manual:.4f} / 함수 결과: {t_func:.4f}")
print(f"두 값이 일치하는가: {np.isclose(t_manual, t_func)}")

# 9. 귀무가설 기각 여부
if p_value < alpha:
    print("p-value < 0.05 → 귀무가설을 기각합니다. 모집단 평균은 180,000달러와 다르다고 볼 수 있습니다.")
else:
    print("p-value >= 0.05 → 귀무가설을 기각하지 못합니다. 모집단 평균이 180,000달러와 다르다고 볼 근거가 부족합니다.")



표본 수: 30
표본평균: 233323.8000
표본표준편차: 86013.0117
표준오차: 15703.7556
직접 계산한 t통계량: 3.3956

Shapiro-Wilk 통계량: 0.9538, p-value: 0.2133
p-value >= 0.05 → 정규성 가정을 기각하지 못함 (정규성 충족으로 봄)

ttest_1samp t통계량: 3.3956
p-value: 0.0020

직접 계산: 3.3956 / 함수 결과: 3.3956
두 값이 일치하는가: True
p-value < 0.05 → 귀무가설을 기각합니다. 모집단 평균은 180,000달러와 다르다고 볼 수 있습니다.


### 필수 1 답변 작성란

- **Q1.**
-> 관측된 평균과 기준값의 차이를 ( 그 차이의 불확실성을 의미하는 )표준 오차로 나눈 값
- **Q2.**
-> 분모가 (표준 오차)가 작아지므로 T통계량의 절대값은 커진다.
- **Q3.**
->일치한다. 두방법 모두 약 3.3956으로 계산됨
- **Q4.**
-> 맞다  P-value 0.002로 0.05보다 작으므로 귀무가설을 기각한다 평균 판매가격은 180,000달러와 다르다.


---

## 필수 2. 독립표본 t검정의 가정 점검과 수행

### 문제 2-1. 주방 품질 `Gd`와 `TA` 집단의 평균 판매가격 비교

#### 문제 설명

주방 품질이 `Gd`인 주택과 `TA`인 주택에서 각각 20개를 추출해 평균 판매가격을 비교합니다. 두 집단은 서로 다른 주택으로 구성되어 있습니다.

#### 요구사항

1. `KitchenQual == "Gd"`와 `KitchenQual == "TA"` 집단의 `SalePrice`에서 각각 `n=20`, `random_state=42`로 표본을 추출하세요.
2. 두 집단의 표본 수와 평균을 확인하세요.
3. 데이터 수집 구조를 근거로 두 집단의 독립성을 설명하세요.
4. 각 집단에 Shapiro-Wilk 정규성 검정을 수행하세요.
5. 두 집단에 Levene 등분산 검정을 수행하세요.
6. 가정 점검 결과에 따라 `equal_var=True` 또는 `equal_var=False`를 결정하세요.
7. `stats.ttest_ind()`로 독립표본 t검정을 수행하세요.
8. t통계량과 p-value를 출력하고 두 집단 평균 차이를 해석하세요.

#### 해석 질문

**Q1.** 두 집단이 독립표본인 이유는 무엇인가요?  
**Q2.** 독립성은 별도의 p-value로 확인할 수 있나요?  
**Q3.** 정규성과 등분산성 가정은 각각 충족되나요?  
**Q4.** `equal_var`에는 어떤 값을 사용해야 하나요?  
**Q5.** 두 집단의 평균 판매가격에는 통계적으로 유의한 차이가 있나요?

#### 제출 결과

- 집단별 표본 수와 평균
- 독립성 설명
- 정규성 및 등분산성 검정 결과
- `equal_var` 선택 근거
- 독립표본 t검정 결과와 해석
- Q1~Q5 답변

In [4]:
# 필수 2 코드를 작성하세요.


# 1. 집단별 SalePrice에서 n=20, random_state=42로 표본 추출
group_gd = df.loc[df["KitchenQual"] == "Gd", "SalePrice"].dropna().sample(n=20, random_state=42)
group_ta = df.loc[df["KitchenQual"] == "TA", "SalePrice"].dropna().sample(n=20, random_state=42)

# 2. 표본 수와 평균
print(f"Gd 집단 - 표본 수: {len(group_gd)}, 평균: {group_gd.mean():.4f}")
print(f"TA 집단 - 표본 수: {len(group_ta)}, 평균: {group_ta.mean():.4f}")

# 3. 독립성 설명
# - 각 주택은 KitchenQual 값을 하나만 가지므로, 한 주택이 Gd와 TA 집단에 동시에 속할 수 없다.
# - 두 표본은 서로 다른 주택 집합에서 각각 따로 추출했으며,
#   한 집단의 관측값이 다른 집단의 특정 관측값과 짝지어져 있지 않다 (대응표본 아님).
# - 따라서 한 집단에서 어떤 주택이 뽑히는지가 다른 집단의 추출 결과에 영향을 주지 않으므로
#   두 집단은 독립이라고 볼 수 있다.

alpha = 0.05

# 4. Shapiro-Wilk 정규성 검정 (H0: 정규분포를 따른다)
_, p_shapiro_gd = stats.shapiro(group_gd)
_, p_shapiro_ta = stats.shapiro(group_ta)
print(f"\nShapiro-Wilk p-value (Gd): {p_shapiro_gd:.4f} → 정규성 충족: {p_shapiro_gd >= alpha}")
print(f"Shapiro-Wilk p-value (TA): {p_shapiro_ta:.4f} → 정규성 충족: {p_shapiro_ta >= alpha}")

# 5. Levene 등분산 검정 (H0: 두 집단의 분산이 같다)
_, p_levene = stats.levene(group_gd, group_ta)
print(f"Levene p-value: {p_levene:.4f} → 등분산성 충족: {p_levene >= alpha}")

# 6. equal_var 결정
equal_var = p_levene >= alpha
print(f"\nequal_var = {equal_var} ({'독립표본 t검정' if equal_var else 'Welch t검정'})")

if p_shapiro_gd < alpha or p_shapiro_ta < alpha:
    print("※ 정규성 가정이 위반된 집단이 있으므로 t검정 결과는 주의해서 해석해야 합니다.")

# 7. 독립표본 t검정 (H0: 두 집단의 평균이 같다 / H1: 다르다)
t_stat, p_value = stats.ttest_ind(group_gd, group_ta, equal_var=equal_var)

# 8. 결과 출력 및 해석
print(f"\nt통계량: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

if p_value < alpha:
    print("p-value < 0.05 → 귀무가설 기각: Gd 집단과 TA 집단의 평균 판매가격 차이는 통계적으로 유의합니다.")
else:
    print("p-value >= 0.05 → 귀무가설 기각 못함: 두 집단의 평균 판매가격이 다르다고 볼 근거가 부족합니다.")

Gd 집단 - 표본 수: 20, 평균: 191916.6000
TA 집단 - 표본 수: 20, 평균: 137432.5000

Shapiro-Wilk p-value (Gd): 0.4580 → 정규성 충족: True
Shapiro-Wilk p-value (TA): 0.2503 → 정규성 충족: True
Levene p-value: 0.0593 → 등분산성 충족: True

equal_var = True (독립표본 t검정)

t통계량: 3.4249
p-value: 0.0015
p-value < 0.05 → 귀무가설 기각: Gd 집단과 TA 집단의 평균 판매가격 차이는 통계적으로 유의합니다.


### 필수 2 답변 작성란

- **Q1.**
-> GD/TA 각 집단이 서로 다른 주택으로 구성되어 하나의 관측값이 다른 관측값과 짝을 이루지 않기 때문이다.

- **Q2.**
-> 일반적으로 독립성은 별도의 검정 P-value 로 확인하는 것이 아니라
-> 데이터 수집 구조, Feature 엔지니어링 구조를 통해 판단한다.
- **Q3.**
필수 2 문제에서는 정규성과 등 분산성 위반 근거가 없다.
- **Q4.**
Levene p-value 가 0.05보다 크므로 True를 사용한다.
- **Q5.**
유의한 차이가 있다, t 검정 결과 p-value 가 약 0.0015로 0.05보다 작다.
일반적으로 주방설계를 잘해놓으면 주택 가격이 달라질 가능성이 있다.


---

## 과제. 대응표본과 독립표본 상황 구분

### 문제 3-1. 같은 주택의 보수 전후 예상 판매가격 비교

#### 문제 설명

부동산 회사가 동일한 주택 10채에 대해 보수 전 예상 판매가격과 보수 후 예상 판매가격을 각각 산정했습니다. 같은 위치의 값은 동일한 주택의 전후 가격으로 서로 짝을 이룹니다.

```python
before_price = [145, 162, 178, 155, 190, 172, 168, 181, 159, 175]
after_price  = [154, 170, 185, 164, 201, 179, 176, 190, 168, 183]
```

단위는 천 달러입니다.

#### 요구사항

1. 두 배열의 길이가 같은지 확인하세요.
2. 보수 전후 평균을 계산하세요.
3. 독립표본 t검정과 대응표본 t검정 중 적절한 방법을 선택하고 이유를 설명하세요.
4. 대응표본 t검정의 정규성 가정은 개별 배열이 아니라 `after_price - before_price` 차이값에 적용된다는 점을 확인하세요.
5. 차이값에 Shapiro-Wilk 정규성 검정을 수행하세요.
6. 적절한 t검정을 실행하고 t통계량과 p-value를 출력하세요.
7. 보수 전후 평균 예상 판매가격에 유의한 차이가 있는지 결론을 작성하세요.

#### 해석 질문

**Q1.** 이 데이터는 독립표본인가요, 대응표본인가요?  
**Q2.** 대응표본 t검정에서는 무엇의 정규성을 확인해야 하나요?  
**Q3.** `stats.ttest_ind()`가 아니라 어떤 함수를 사용해야 하나요?  
**Q4.** 검정 결과 보수 전후 예상 판매가격에 유의한 차이가 있나요?

#### 제출 결과

- 표본 관계 판단과 근거
- 전후 평균과 차이값
- 차이값의 정규성 결과
- 대응표본 t검정 결과
- 결과 해석
- Q1~Q4 답변

In [2]:
# 과제 코드를 작성하세요.
import numpy as np
from scipy import stats

before_price = np.array([145, 162, 178, 155, 190, 172, 168, 181, 159, 175])
after_price  = np.array([154, 170, 185, 164, 201, 179, 176, 190, 168, 183])

# 1) 길이 확인
print(f"before 길이 = {len(before_price)}, after 길이 = {len(after_price)}")
print(f"길이 일치: {len(before_price) == len(after_price)}")

# 2) 전후 평균
print(f"\n보수 전 평균 = {before_price.mean():.4f}")
print(f"보수 후 평균 = {after_price.mean():.4f}")

# 4) 차이값 생성
diff = after_price - before_price
print(f"\n차이값 = {diff}")
print(f"차이값 평균 = {diff.mean():.4f}")

# 5) 차이값에 Shapiro-Wilk 정규성 검정
sw = stats.shapiro(diff)
print(f"\n[정규성] W = {sw.statistic:.4f}, p = {sw.pvalue:.4f}")
print(f"→ {'정규성 충족' if sw.pvalue > 0.05 else '정규성 위배'}")

# 6) 대응표본 t검정
t_stat, p_value = stats.ttest_rel(after_price, before_price)
print(f"\nt-통계량 = {t_stat:.4f}")
print(f"p-value  = {p_value:.4f}  (지수: {p_value:.4e})")
print(f"→ {'유의한 차이 있음' if p_value <= 0.05 else '유의한 차이 없음'}")

before 길이 = 10, after 길이 = 10
길이 일치: True

보수 전 평균 = 168.5000
보수 후 평균 = 177.0000

차이값 = [ 9  8  7  9 11  7  8  9  9  8]
차이값 평균 = 8.5000

[정규성] W = 0.8871, p = 0.1574
→ 정규성 충족

t-통계량 = 22.8079
p-value  = 0.0000  (지수: 2.8420e-09)
→ 유의한 차이 있음


### 과제 답변 작성란

**Q1.** 이 데이터는 독립표본인가요, 대응표본인가요? 
이 데이터는 대응 표본입니다. 왜냐하면 동일한 주택 10채를 보수 전과 후에 각각 측정 했고, 같은 위치의 값이 같은 주택의 대응을 이루므로 두 배열은 서로 독립이 아니다.

**Q2.** 대응표본 t검정에서는 무엇의 정규성을 확인해야 하나요? 
대응 표본 t검정에서는  차이값의 정규성을 확인해야한다. 대응표본 t 검정은 차이 값에 대한 단일 표본 t 검정과 수학적으로 동일하기 때문에, 검정이 실제로 다루는 대상이 차이값입니다.


**Q3.** `stats.ttest_ind()`가 아니라 어떤 함수를 사용해야 하나요?  
stat.ttest_rel(after_price, before_price)를 사용한다. 


**Q4.** 검정 결과 보수 전후 예상 판매가격에 유의한 차이가 있나요?
유의한 차이가 있다 왜냐하면 p value 0.05보다 낮기 때문에.


---

## 실습 마무리

1. t통계량은 어떤 두 값을 비교하여 계산하나요?
-> 관측된 평균 차이를 표준 오차와 비교한다. 
2. 단일표본 t검정은 어떤 상황에서 사용하나요?
3. 독립표본과 대응표본은 데이터 수집 구조에서 어떤 차이가 있나요?
4. 독립표본 t검정에서 정규성·등분산성·독립성은 각각 어떻게 확인하나요?
5. 분산이 같다고 보기 어렵거나 확신할 수 없을 때 어떤 t검정을 사용할 수 있나요?